In [1]:
import sys
sys.path.append('../')

import scqubits as scq
import pandas as pd
import qutip as qt
import numpy as np
from matplotlib import pyplot as plt
from qutip.qip.operations import rz, cz_gate
import cmath
from tqdm import tqdm
from matplotlib.colors import LogNorm
import datetime
import pytz
import scqubits.settings as settings
settings.OVERLAP_THRESHOLD = 0.3
from joblib import Parallel, delayed
import itertools
import scipy.sparse as ssp
from sympy import symbols
import scipy as sp
import utils_2Q_gate_zp as ut
import os
from datetime import datetime
from multiprocessing import Pool

In [5]:
drive_phi, drive_theta, truc = False, True, 150
drive_0 = True
folder = 'data_xgate_theta_3ncut.txt' if drive_theta else 'data_xgate_phi_3ncut.txt'
f_xgate = pd.read_csv('data/'+folder)
params = f_xgate[['tg', 'drive_amp_1', 'drive_amp_2', 'detune_1', 'detune_2'
                    ]].to_numpy()[[-5], :]

num_cpus, n_job = 4, 1*len(params)
logi_state = [0, 2]

truncation=1000
folder = '../../data/3ncut_one_zeropi/'
if drive_0:
    evals = 2*np.pi* scq.read(folder + f'zeropi_0_specdata_truc={truncation}_3ncut.h5').energy_table
    n_theta = 2*np.pi* scq.read(folder + f'zeropi_0_n_theta_truc={truncation}_3ncut.h5').matrixelem_table
    n_phi = 2*np.pi* scq.read(folder + f'zeropi_0_n_phi_truc={truncation}_3ncut.h5').matrixelem_table
else:
    evals = 2*np.pi* scq.read(folder + f'zeropi_1_specdata_truc={truncation}_3ncut.h5').energy_table
    n_theta = 2*np.pi* scq.read(folder + f'zeropi_1_n_theta_truc={truncation}_3ncut.h5').matrixelem_table
    n_phi = 2*np.pi* scq.read(folder + f'zeropi_1_n_phi_truc={truncation}_3ncut.h5').matrixelem_table

evals = evals - evals[0]
gate_target = qt.sigmax()

H0 = qt.Qobj(np.diag(evals))
if drive_phi:
    w_trans_1 = evals[9] - evals[0]
    w_trans_2 = evals[9] - evals[2]
    drive_term = n_phi
if drive_theta:
    w_trans_1 = evals[7] - evals[0]
    w_trans_2 = evals[7] - evals[2]
    drive_term = n_theta

thresh = 0.01
hspace_charge = [0, 2]  # Start with the ground and first excited states
for s in hspace_charge:
    for i in range(truc):
        if np.abs(drive_term[s, i] / (2 * np.pi)) > thresh and i not in hspace_charge:
            hspace_charge.append(i)
hspace_charge.sort()
# hspace_charge = np.arange(truc).tolist()
print('params =')
for para in params:
    print(para.tolist(), ',')

params =
[79.950621, 0.050323, 0.013699, 0.001008, 0.001432] ,


In [8]:
t1_other = 5 # μs

tphi_logi = 100 # μs
gamma_decay_logi =  1 / 1600e3
gamma_dephase_logi = 1 / 1e3 / tphi_logi
gamma_decay_other =  1 / 1e3 / t1_other
gamma_dephase_other = 1 / 1e3 / t1_other
idx_2 = 2 if drive_theta else 1

hspace_len = len(hspace_charge)
if drive_theta:
    gamma_decay_old   = [0, gamma_decay_other,  gamma_decay_logi]  + [gamma_decay_other]  * (hspace_len-3)
    gamma_dephase_old = [0, gamma_dephase_other, gamma_dephase_logi] + [gamma_dephase_other] * (hspace_len-3)
else:
    gamma_decay_old   = [0,  gamma_decay_logi]  + [gamma_decay_other]  * (hspace_len-2)
    gamma_dephase_old = [0,  gamma_dephase_logi] + [gamma_dephase_other] * (hspace_len-2)

folder_1 = 'data/data_gamma_'
folder_2 = 'theta.txt' if drive_theta else 'phi.txt'
gamma_new = pd.read_csv(folder_1 + folder_2)

### 't1_50us_47', 't1_50us_27', 't1_50us_07'
### 'tphi_50us_02', 'tphi_50us_07', 'tphi_1e6'
gamma_decay_new = gamma_new['t1_50us_47'].to_numpy()
# gamma_dephase_new = gamma_new['tphi_1e6'].to_numpy()
gamma_dephase_new = gamma_new['tphi_50us_02'].to_numpy()

print("gamma_decay_new[2] = ", gamma_decay_new[2], ", gamma_dephase_new[2] = ", gamma_dephase_new[2])
gamma_decay_new = gamma_decay_new *50 /t1_other
gamma_dephase_new = gamma_dephase_new *50 /t1_other

jump_t1   = []
jump_tphi = []
for i in range(1,hspace_len):
    jump_t1.append( np.sqrt(gamma_decay_new[i]) * qt.basis(hspace_len,0) * qt.basis(hspace_len,i).dag() )
    jump_tphi.append( np.sqrt(2*gamma_dephase_new[i]) * qt.basis(hspace_len,i).proj() )

logi_idx = [hspace_charge.index(s) for s in logi_state]
H0_truc = ut.truncate_2(H0, hspace_charge)
drive_truc = ut.truncate_2(drive_term, hspace_charge)
H_qbt_drive = [H0_truc, [drive_truc, ut.drive_gauss_A],
                        [drive_truc, ut.drive_gauss_B],]

gamma_decay_new[2] =  4.4687146634489855e-35 , gamma_dephase_new[2] =  2e-05


In [9]:
############################################################
# c_op_list = [qt.Qobj(np.zeros((truc, truc)))]
# c_op_list = []
# args = [H_qbt_drive, w_trans_1, w_trans_2, num_cpus, c_op_list, logi_idx, gate_target]
# f_ideal = Parallel(n_jobs=n_job)(delayed(ut.xgate_fidelity_log_noise)(args_indep, *args)
#                                             for args_indep in params)
# print('\nf_ideal = [')
# for i in range(0, len(f_ideal), 4):
#     print(', '.join(map(str, f_ideal[i:i+4])), ',')
# print(']')
# print("Current Mountain Time:", datetime.now(pytz.timezone('America/Denver')))

############################################################
c_op_list = jump_t1 + jump_tphi
args = [H_qbt_drive, w_trans_1, w_trans_2, num_cpus, c_op_list, logi_idx, gate_target]
f_noise = Parallel(n_jobs=n_job)(delayed(ut.xgate_fidelity_log_noise)(args_indep, *args)
                                            for args_indep in params)
print('\nf_noise = [')
for i in range(0, len(f_noise), 4):
    print(', '.join(map(str, f_noise[i:i+4])), ',')
print(']')
print("Current Mountain Time:", datetime.now(pytz.timezone('America/Denver')))


f_noise = [
-2.303046445312451 ,
]
Current Mountain Time: 2025-03-04 12:00:50.019268-07:00


In [11]:
np.shape(c_op_list)

(154, 78, 78)

In [ ]:
# np.savez("data/data_xgate_theta_collapse_op.npz", arr1=c_op_list)